In [1]:
%load_ext autoreload
%autoreload 2

# `import`

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from collections import defaultdict

from medical_rl.libs.RL import Model
from medical_rl.libs.envs import get_env
from medical_rl.libs.cluster import get_data_clustered
from medical_rl.data_formatters.amsterdam import AmsterdamFormatter

from dice_rl_TU_Vienna.wrappers import AbsorbingWrapper, LoopingWrapper
from dice_rl_TU_Vienna.dataset import get_dataset_from_df, get_dataset_from_env
from dice_rl_TU_Vienna.estimators.onpolicy import OnPE
from dice_rl_TU_Vienna.utils.pandas import head_by_id
from dice_rl_TU_Vienna.plugins.stable_baselines3.policy import (
    get_model_MaskablePPO, get_probs_MaskablePPO, )

from plugins.sepsis.bologheanu.config import *

/Users/richardweiss/Documents/.venvs/dice_rl_TU_Vienna/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [3]:
pd.set_option("display.max_rows", 100)

# Prepare

In [4]:
# constants continuous

seed = 42
n_act = 5

In [5]:
# constants tabular

# ---------------------------------------------------------------- #

# policies

total_timesteps = {
    "exploratory": 10_000,
    "evaluation": 100_000,
}

# ---------------------------------------------------------------- #
# clustering

seed = 42
n_clusters = 256
n_init = 10

n_obs = n_clusters + 2
n_act = 5

# ---------------------------------------------------------------- #
# dataset

n_trajectories = 10_000
max_trajectory_length = None
# seed

# ---------------------------------------------------------------- #

In [6]:
def RL_split_to_dataset_primal(RL_split, tabular):

    dtype_obs = np.int64 if tabular else np.float32

    split_id, split_t, split_obs, split_act, split_ret = RL_split

    split_id  = np.array(split_id,  dtype=np.int64)
    split_t   = np.array(split_t,   dtype=np.int64)
    split_obs = np.array(split_obs, dtype=dtype_obs)
    split_act = np.array(split_act, dtype=np.int64)
    split_ret = np.array(split_ret, dtype=np.float32)

    id  = []
    t   = []
    obs = []
    act = []
    rew = []

    obs_init = []
    obs_next = []

    i_init = 0; i_next = None
    for i in range( len(split_id) ):

        # first trajectory position
        A = i > 0 and split_id[i] != split_id[i-1]

        # last trajectory position
        B = i == len(split_id) - 1 or split_id[i+1] != split_id[i]

        if A:
            i_init = i

        if B:
            continue
        else:
            i_next = i+1

        # second last trajectory position
        C = i == len(split_id) - 2 or split_id[i+2] != split_id[i]

        id .append(split_id [i])
        t  .append(split_t  [i])
        obs.append(split_obs[i])
        act.append(split_act[i])
        rew.append(split_ret[i] if C else 0)

        obs_init.append( split_obs[i_init] )
        obs_next.append( split_obs[i_next] )

    return pd.DataFrame({
        "id": id, "t": t,
        "obs_init": obs_init,
        "obs": obs, "act": act, "rew": rew,
        "obs_next": obs_next,
    })

In [7]:
def get_processing(processors=None):
    if processors is None: processors = []

    def processing(df):
        df = df.copy(deep=False)
        for processor in processors:
            df = processor(df)
        return df

    return processing

In [8]:
def add_terminal_transition(df):
    df = df.copy(deep=False)

    df_parts = []

    for id in df["id"].unique():
        df_part = df[ df["id"] == id ].reset_index(drop=True)

        row = df_part.loc[len(df_part)-1].copy()
        row["t"] += 1
        row["act"] = np.random.choice( range(n_act), p=row["probs_next"], )
        row["rew"] = 0
        for s in ["obs", "probs", "probs_behavior"]:
            if (s_next := f"{s}_next") in df_part.columns:
                row[s] = row[s_next]
        df_part.loc[len(df_part)] = row

        df_parts.append(df_part)

    df = pd.concat(df_parts).reset_index(drop=True)

    return df

In [ ]:
def assertion_terminal(dataset, i):
    assertion = True
    for s in ["obs", "probs", "probs_benavior"]:
        if s not in dataset.columns: continue

        x = dataset.loc[i,   "probs"]
        y = dataset.loc[i-1, "probs_next"]
        z = dataset.loc[i,   "probs_next"]

        if not np.all(x == y) and np.all(y == z):
            assertion = False

    return assertion

In [10]:
get_OnPV = lambda dataset: OnPE(dataset).solve(gamma=0.9)[0]

In [11]:
test = {}
bounds = {}
model = {}
dataset = {}

In [12]:
def load_datasets_raw():
    train, valid, test =  AmsterdamFormatter() \
        .load_random_split(dir_split)

    datasets_raw = {
        "train": train,
        "valid": valid,
        "test": test,
    }

    return datasets_raw

In [13]:
head_by_id(
    load_datasets_raw()["test"],
    n=2,
    id_name="ID",
)

,ID,Length of Stay,ACTH max,ACTH mean,ACTH min,ACTH std,AF Monitor max,AF Monitor mean,AF Monitor min,AF Monitor std,...,Fluid balance,AdmissionCount,Steroids,SOFA score,Age,Gender,Antibiotics,Antiviral,Antifungal,ICU Mortality
0,20.0,0,-0.080002,-0.088658,-0.088606,-0.025408,-0.094499,-0.078563,-0.081722,-0.164105,...,0.242128,-0.271663,1,-0.769852,3,1,1,0,0,0
1,20.0,1,-0.080002,-0.088658,-0.088606,-0.025408,-0.094499,-0.078563,-0.081722,-0.164105,...,-0.660426,-0.271663,1,-0.050286,3,1,1,0,0,0
2,20.0,2,-0.080002,-0.088658,-0.088606,-0.025408,-0.094499,-0.078563,-0.081722,-0.164105,...,-0.669512,-0.271663,1,-0.290141,3,1,1,0,0,0
3,20.0,3,-0.080002,-0.088658,-0.088606,-0.025408,-0.094499,-0.078563,-0.081722,-0.164105,...,-0.210015,-0.271663,1,-0.529997,3,1,1,0,0,0
4,20.0,4,-0.080002,-0.088658,-0.088606,-0.025408,-0.094499,-0.078563,-0.081722,-0.164105,...,-0.656099,-0.271663,1,-0.529997,3,1,1,0,0,0
5,20.0,5,-0.080002,-0.088658,-0.088606,-0.025408,-0.094499,-0.078563,-0.081722,-0.164105,...,-0.960701,-0.271663,1,-1.489417,3,1,1,0,0,0
6,20.0,6,-0.080002,-0.088658,-0.088606,-0.025408,-0.094499,-0.078563,-0.081722,-0.164105,...,-0.540143,-0.271663,1,-1.489417,3,1,1,0,0,0
7,20.0,7,-0.080002,-0.088658,-0.088606,-0.025408,-0.094499,-0.078563,-0.081722,-0.164105,...,0.154728,-0.271663,0,-0.529997,3,1,4,0,0,0
8,25.0,0,-0.080002,-0.088658,-0.088606,-0.025408,-0.238616,-0.310907,-0.819176,-0.542989,...,-0.154200,-0.271663,2,1.388845,4,2,0,0,0,1
9,25.0,1,-0.080002,-0.088658,-0.088606,-0.025408,0.121676,-0.039660,-1.451279,-0.724618,...,0.045695,-0.271663,2,1.388845,4,2,0,0,0,1


# Tabular

## Dataset

In [29]:
hyperparameters={
    "seed": seed,
    "features": [
        'RRmean (ABP) mean',
        'RRmean (NIBP) mean',
        'Leukozyten max',
        'Heartrate mean',
        'Heartrate std',
        'Glucose mean',
        'Respiratory Rate resp min',
        'Noradrenaline (Norepinefrine) max',
        'Thrombozyten max',
        'PTT max',
        'PEEP max',
        'Fluid balance',
        'SOFA score',
        'Age',
        'Gender',
        'Antibiotics',
    ],
    "n_clusters": n_clusters,
    "n_init": n_init,
}

In [30]:
def dataset_raw_to_dataset_clustered_raw(
        dataset_raw, hyperparameters):

    dataset_clustered_raw = get_data_clustered(
        data=dataset_raw,
        features=hyperparameters["features"],
        n_clusters=hyperparameters["n_clusters"],
        n_init=hyperparameters["n_init"],
        seed=hyperparameters["seed"],
        path=dir_clustering,
    )

    return dataset_clustered_raw

In [31]:
head_by_id(
    dataset_raw_to_dataset_clustered_raw(
        load_datasets_raw()["test"],
        hyperparameters,
    ),
    n=2,
)

Trying to load data/medical_rl/sepsis_bologheanu/2025-01-30T11:42:45.694984/2025-01-30T12:01:58.273325/kmeans_model.pkl


,id,t,obs,act,ret
0,20,0,58,1,0
1,20,1,179,1,0
2,20,2,17,1,0
3,20,3,17,1,0
4,20,4,17,1,0
5,20,5,17,1,0
6,20,6,17,1,0
7,20,7,256,0,0
8,25,0,254,2,1
9,25,1,254,2,1


In [32]:
def dataset_clustered_raw_to_RL_split(df):
    return df["id"], df["t"], df["obs"], df["act"], df["ret"]

In [33]:
def dataset_clustered_raw_to_dataset_clustered_primal(dataset_clustered_raw):
    RL_split = dataset_clustered_raw_to_RL_split(dataset_clustered_raw)
    dataset_clustered_primal = RL_split_to_dataset_primal(RL_split, tabular=True)

    return dataset_clustered_primal

In [34]:
head_by_id(
    dataset_clustered_raw_to_dataset_clustered_primal(
        dataset_raw_to_dataset_clustered_raw(
            load_datasets_raw()["test"],
            hyperparameters,
        ),
    ),
    n=2,
)

Trying to load data/medical_rl/sepsis_bologheanu/2025-01-30T11:42:45.694984/2025-01-30T12:01:58.273325/kmeans_model.pkl


,id,t,obs_init,obs,act,rew,obs_next
0,20,0,58,58,1,0.0,179
1,20,1,58,179,1,0.0,17
2,20,2,58,17,1,0.0,17
3,20,3,58,17,1,0.0,17
4,20,4,58,17,1,0.0,17
5,20,5,58,17,1,0.0,17
6,20,6,58,17,1,0.0,256
7,25,0,254,254,2,0.0,254
8,25,1,254,254,2,1.0,257


In [35]:
def get_datasets_tabular_until_primal():
    datasets_raw = load_datasets_raw()
    datasets_clustered_raw = {
        k: dataset_raw_to_dataset_clustered_raw(v, hyperparameters)
            for k, v in datasets_raw.items()
    }
    datasets_clustered_primal = {
        k: dataset_clustered_raw_to_dataset_clustered_primal(v)
            for k, v in datasets_clustered_raw.items()
    }

    datasets_tabular = {
        "raw": datasets_raw,
        "clustered_raw": datasets_clustered_raw,
        "clustered_primal": datasets_clustered_primal
    }

    return datasets_tabular

In [36]:
dataset["tabular"] = get_datasets_tabular_until_primal()

Trying to load data/medical_rl/sepsis_bologheanu/2025-01-30T11:42:45.694984/2025-01-30T12:01:58.273325/kmeans_model.pkl
Trying to load data/medical_rl/sepsis_bologheanu/2025-01-30T11:42:45.694984/2025-01-30T12:01:58.273325/kmeans_model.pkl
Trying to load data/medical_rl/sepsis_bologheanu/2025-01-30T11:42:45.694984/2025-01-30T12:01:58.273325/kmeans_model.pkl


In [37]:
for label in labels:
    dataset["tabular"]["clustered_primal"][label].to_parquet(
        os.path.join(dir_clustering, f"{label}.parquet"), )

In [38]:
env = { label: {} for label in labels }
action_mask = {}

for label in labels:
    x, y = get_env(
        n_clusters=n_clusters,
        data_clustered=dataset["tabular"]["clustered_primal"][label],
    )

    env[label] = {
        "": x,
        "absorbing": AbsorbingWrapper(x),
        "looping": LoopingWrapper(x),
    }
    action_mask[label] = y

In [ ]:
model["tabular"] = {}
id_policy["tabular"] = {} # type: ignore

for name in names:
    if name == "original": continue

    x, y = get_model_MaskablePPO(
        dir_data=dir_clustering,
        env=env["train"][""],
        hyperparameters={
            "seed": seed,
            "total_timesteps": total_timesteps[name],
        },
    )

    model["tabular"][name] = x
    id_policy["tabular"][name] = y

def get_act_(name, obs):
    with warnings.catch_warnings(action="ignore", category=UserWarning):
        act, _ = model["tabular"][name].predict(obs, action_masks=action_mask["test"][obs])
        act = int(act)
        return act

get_act_exploratory = lambda obs: get_act_("exploratory", obs)
get_act_evaluation  = lambda obs: get_act_("evaluation",  obs)

get_act = {
    "exploratory": get_act_exploratory,
    "evaluation": get_act_evaluation,
}

dir_policy["tabular"] = {
    name: os_path_join(dir_clustering, id_policy["tabular"][name])
        for name in names
            if name != "original"
}

In [40]:
# final processing

def get_probs_behavior_dict(df):

    mults = defaultdict(lambda: [ 0 for act in range(n_act) ] )

    for i in range( len(df) ):
        obs = df.loc[i, "obs"]
        act = df.loc[i, "act"]
        mults[obs][act] += 1 # type: ignore

    probs = {
        k: np.array( v / np.sum(v), dtype=np.float32, )
        for k, v in mults.items()
    }

    return probs

def add_probs_tabular_evaluation(df):
    df = df.copy(deep=False)
    df["probs_init"] = list( get_probs_MaskablePPO(df["obs_init"], model["tabular"]["evaluation"], action_mask["test"]) )
    df["probs"]      = list( get_probs_MaskablePPO(df["obs"],      model["tabular"]["evaluation"], action_mask["test"]) )
    df["probs_next"] = list( get_probs_MaskablePPO(df["obs_next"], model["tabular"]["evaluation"], action_mask["test"]) )
    return df

def add_probs_tabular_behavior_original(df):
    df = df.copy(deep=False)
    probs_behavior_dict = get_probs_behavior_dict(df)
    df["probs_behavior_init"] = df["obs_init"].map(probs_behavior_dict)
    df["probs_behavior"]      = df["obs"]     .map(probs_behavior_dict)
    df["probs_behavior_next"] = df["obs_next"].map(probs_behavior_dict)
    uniform = np.ones(n_act, dtype=np.float32) / n_act
    df = df.map(lambda x: x if isinstance(x, np.ndarray) or not pd.isna(x) else uniform)
    return df

def add_probs_tabular_behavior_exploratory(df):
    df = df.copy(deep=False)
    df["probs_behavior_init"] = list( get_probs_MaskablePPO(df["obs_init"], model["tabular"]["exploratory"], action_mask["test"]) )
    df["probs_behavior"]      = list( get_probs_MaskablePPO(df["obs"],      model["tabular"]["exploratory"], action_mask["test"]) )
    df["probs_behavior_next"] = list( get_probs_MaskablePPO(df["obs_next"], model["tabular"]["exploratory"], action_mask["test"]) )
    return df

def cast_probs(df):
    df = df.copy(deep=False)
    cast_array = lambda x: np.array(x, dtype=np.float32)

    for column in df.columns:
        if "probs" not in column: continue

        df[column] = df[column].apply(cast_array) # type: ignore

    return df


dataset_clustered_primal_to_dataset_clustered_final = {
    "original": get_processing([
        add_probs_tabular_evaluation,
        add_probs_tabular_behavior_original,
        add_terminal_transition,
    ]),
    "exploratory": get_processing([
        add_probs_tabular_evaluation,
        add_probs_tabular_behavior_exploratory,
        add_terminal_transition,
    ]),
    "evaluation": get_processing([
        add_probs_tabular_evaluation,
        add_terminal_transition,
    ]),
}

In [ ]:
def get_dataset_clustered_final_(name):
    if name == "original":
        return get_dataset_from_df(
            dir=dir_clustering,
            df=dataset["tabular"]["clustered_primal"]["test"],
            hyperparameters={
                "evaluation_policy": {
                    "id": id_policy["tabular"]["evaluation"], # type: ignore
                },
            },
            postprocessing=dataset_clustered_primal_to_dataset_clustered_final[name],
            verbosity=1,
        )

    if name in ["exploratory", "evaluation"]:
        return get_dataset_from_env(
            dir=dir_policy["tabular"][name],
            env=env["test"][""],
            get_act=get_act[name],
            hyperparameters={
                "n_trajectories": n_trajectories,
                "max_trajectory_length": max_trajectory_length,
                "seed": seed,
                "evaluation_policy": {
                    "id": id_policy["tabular"]["evaluation"], # type: ignore
                },
            },
            postprocessing=dataset_clustered_primal_to_dataset_clustered_final[name],
            verbosity=1,
        )
    
    raise ValueError

def get_dataset_clustered_final():
    dataset_clustered_final = {}
    id_dataset = {}

    for name in names:
        x, y = get_dataset_clustered_final_(name)
        dataset_clustered_final[name] = x
        id_dataset[name] = y

    return dataset_clustered_final, id_dataset

In [42]:
dataset["tabular"]["clustered_final"], id_dataset["tabular"] = get_dataset_clustered_final()

trying to find id_dataset in data/medical_rl/sepsis_bologheanu/2025-01-30T11:42:45.694984/2025-01-30T12:01:58.273325/dataset.json
trying to load dataset from data/medical_rl/sepsis_bologheanu/2025-01-30T11:42:45.694984/2025-01-30T12:01:58.273325/2025-04-04T14:36:57.598315/dataset.parquet
trying to find id_dataset in data/medical_rl/sepsis_bologheanu/2025-01-30T11:42:45.694984/2025-01-30T12:01:58.273325/2025-04-04T14:16:14.383206/dataset.json
trying to load dataset from data/medical_rl/sepsis_bologheanu/2025-01-30T11:42:45.694984/2025-01-30T12:01:58.273325/2025-04-04T14:16:14.383206/2025-04-04T14:37:38.614841/dataset.parquet
trying to find id_dataset in data/medical_rl/sepsis_bologheanu/2025-01-30T11:42:45.694984/2025-01-30T12:01:58.273325/2025-04-04T14:20:10.762120/dataset.json
trying to load dataset from data/medical_rl/sepsis_bologheanu/2025-01-30T11:42:45.694984/2025-01-30T12:01:58.273325/2025-04-04T14:20:10.762120/2025-04-04T14:40:16.046389/dataset.parquet


In [43]:
for k, v in dataset["tabular"]["clustered_final"].items():
    print(f"{k=}")
    display(head_by_id(v, n=2))

k='original'


,id,t,obs_init,obs,act,rew,obs_next,probs_init,probs,probs_next,probs_behavior_init,probs_behavior,probs_behavior_next
0,20,0,58,58,1,0.0,179,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.0009460958, 0.98486024, 0.00033117103, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.5987261, 0.18152866, 0.05732484, 0.10509554..."
1,20,1,58,179,1,0.0,17,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.0009460958, 0.98486024, 0.00033117103, 0.00...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.5987261, 0.18152866, 0.05732484, 0.10509554...","[0.64451313, 0.1576507, 0.038639877, 0.1112828..."
2,20,2,58,17,1,0.0,17,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64451313, 0.1576507, 0.038639877, 0.1112828...","[0.64451313, 0.1576507, 0.038639877, 0.1112828..."
3,20,3,58,17,1,0.0,17,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64451313, 0.1576507, 0.038639877, 0.1112828...","[0.64451313, 0.1576507, 0.038639877, 0.1112828..."
4,20,4,58,17,1,0.0,17,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64451313, 0.1576507, 0.038639877, 0.1112828...","[0.64451313, 0.1576507, 0.038639877, 0.1112828..."
5,20,5,58,17,1,0.0,17,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64451313, 0.1576507, 0.038639877, 0.1112828...","[0.64451313, 0.1576507, 0.038639877, 0.1112828..."
6,20,6,58,17,1,0.0,256,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.099401385, 0.08730502, 0.018886177, 0.36740...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64451313, 0.1576507, 0.038639877, 0.1112828...","[0.2, 0.2, 0.2, 0.2, 0.2]"
7,20,7,58,256,4,0.0,256,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.099401385, 0.08730502, 0.018886177, 0.36740...","[0.099401385, 0.08730502, 0.018886177, 0.36740...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.2, 0.2, 0.2, 0.2, 0.2]","[0.2, 0.2, 0.2, 0.2, 0.2]"
8,25,0,254,254,2,0.0,254,"[0.03472524, 0.033255864, 0.9320189, 0.0, 0.0]","[0.03472524, 0.033255864, 0.9320189, 0.0, 0.0]","[0.03472524, 0.033255864, 0.9320189, 0.0, 0.0]","[0.6666667, 0.16666667, 0.16666667, 0.0, 0.0]","[0.6666667, 0.16666667, 0.16666667, 0.0, 0.0]","[0.6666667, 0.16666667, 0.16666667, 0.0, 0.0]"
9,25,1,254,254,2,1.0,257,"[0.03472524, 0.033255864, 0.9320189, 0.0, 0.0]","[0.03472524, 0.033255864, 0.9320189, 0.0, 0.0]","[0.019111522, 0.31436116, 0.07228454, 0.422952...","[0.6666667, 0.16666667, 0.16666667, 0.0, 0.0]","[0.6666667, 0.16666667, 0.16666667, 0.0, 0.0]","[0.2, 0.2, 0.2, 0.2, 0.2]"


k='exploratory'


,id,t,obs_init,obs,act,rew,obs_next,probs_init,probs,probs_next,probs_behavior_init,probs_behavior,probs_behavior_next
0,0,0,97,97,1,0,42,"[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.010555702, 0.26572725, 0.5671722, 0.1290787...","[0.1389955, 0.20588483, 0.41373006, 0.15858203...","[0.1389955, 0.20588483, 0.41373006, 0.15858203...","[0.11260244, 0.33042437, 0.27037343, 0.1280402..."
1,0,1,97,42,3,1,257,"[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.010555702, 0.26572725, 0.5671722, 0.1290787...","[0.019111522, 0.31436116, 0.07228454, 0.422952...","[0.1389955, 0.20588483, 0.41373006, 0.15858203...","[0.11260244, 0.33042437, 0.27037343, 0.1280402...","[0.16155878, 0.2706891, 0.13586152, 0.22651583..."
2,0,2,97,257,3,0,257,"[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.019111522, 0.31436116, 0.07228454, 0.422952...","[0.019111522, 0.31436116, 0.07228454, 0.422952...","[0.1389955, 0.20588483, 0.41373006, 0.15858203...","[0.16155878, 0.2706891, 0.13586152, 0.22651583...","[0.16155878, 0.2706891, 0.13586152, 0.22651583..."
3,1,0,144,144,0,0,144,"[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.36340088, 0.14329387, 0.11220293, 0.2480995...","[0.36340088, 0.14329387, 0.11220293, 0.2480995...","[0.36340088, 0.14329387, 0.11220293, 0.2480995..."
4,1,1,144,144,1,0,144,"[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.36340088, 0.14329387, 0.11220293, 0.2480995...","[0.36340088, 0.14329387, 0.11220293, 0.2480995...","[0.36340088, 0.14329387, 0.11220293, 0.2480995..."
5,1,2,144,144,0,0,247,"[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.0012141267, 0.0132386, 0.6461953, 0.2690848...","[0.36340088, 0.14329387, 0.11220293, 0.2480995...","[0.36340088, 0.14329387, 0.11220293, 0.2480995...","[0.08255359, 0.22748806, 0.25635046, 0.2646325..."
6,1,3,144,247,0,0,44,"[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.0012141267, 0.0132386, 0.6461953, 0.2690848...","[0.01352798, 0.012362986, 0.007120129, 0.82230...","[0.36340088, 0.14329387, 0.11220293, 0.2480995...","[0.08255359, 0.22748806, 0.25635046, 0.2646325...","[0.13159919, 0.16863076, 0.13916346, 0.3634553..."
7,1,4,144,44,3,0,247,"[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.01352798, 0.012362986, 0.007120129, 0.82230...","[0.0012141267, 0.0132386, 0.6461953, 0.2690848...","[0.36340088, 0.14329387, 0.11220293, 0.2480995...","[0.13159919, 0.16863076, 0.13916346, 0.3634553...","[0.08255359, 0.22748806, 0.25635046, 0.2646325..."
8,1,5,144,247,2,0,44,"[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.0012141267, 0.0132386, 0.6461953, 0.2690848...","[0.01352798, 0.012362986, 0.007120129, 0.82230...","[0.36340088, 0.14329387, 0.11220293, 0.2480995...","[0.08255359, 0.22748806, 0.25635046, 0.2646325...","[0.13159919, 0.16863076, 0.13916346, 0.3634553..."
9,1,6,144,44,1,0,44,"[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.01352798, 0.012362986, 0.007120129, 0.82230...","[0.01352798, 0.012362986, 0.007120129, 0.82230...","[0.36340088, 0.14329387, 0.11220293, 0.2480995...","[0.13159919, 0.16863076, 0.13916346, 0.3634553...","[0.13159919, 0.16863076, 0.13916346, 0.3634553..."


k='evaluation'


,id,t,obs_init,obs,act,rew,obs_next,probs_init,probs,probs_next
0,0,0,97,97,2,0,104,"[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.034258768, 0.008222747, 0.0464423, 0.911076..."
1,0,1,97,104,3,0,24,"[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.034258768, 0.008222747, 0.0464423, 0.911076...","[0.0048171766, 0.91280186, 0.07493122, 0.00744..."
2,0,2,97,24,1,0,122,"[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.0048171766, 0.91280186, 0.07493122, 0.00744...","[0.0018074316, 0.86823225, 0.09632578, 0.03036..."
3,0,3,97,122,1,0,244,"[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.0018074316, 0.86823225, 0.09632578, 0.03036...","[0.007445688, 0.09870003, 0.7324922, 0.0280044..."
4,0,4,97,244,4,0,244,"[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.007445688, 0.09870003, 0.7324922, 0.0280044...","[0.007445688, 0.09870003, 0.7324922, 0.0280044..."
5,0,5,97,244,2,0,48,"[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.007445688, 0.09870003, 0.7324922, 0.0280044...","[0.07213532, 0.022990702, 0.15267873, 0.040649..."
6,0,6,97,48,4,0,256,"[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.07213532, 0.022990702, 0.15267873, 0.040649...","[0.099401385, 0.08730502, 0.018886177, 0.36740..."
7,0,7,97,256,3,0,256,"[0.023121499, 0.018567674, 0.5829504, 0.041359...","[0.099401385, 0.08730502, 0.018886177, 0.36740...","[0.099401385, 0.08730502, 0.018886177, 0.36740..."
8,1,0,144,144,2,0,247,"[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.0012141267, 0.0132386, 0.6461953, 0.2690848..."
9,1,1,144,247,2,0,247,"[0.077239595, 0.26684684, 0.2533386, 0.4010225...","[0.0012141267, 0.0132386, 0.6461953, 0.2690848...","[0.0012141267, 0.0132386, 0.6461953, 0.2690848..."


In [44]:
display(
    assertion_terminal(dataset["tabular"]["clustered_final"]["original"], 7) and \
    assertion_terminal(dataset["tabular"]["clustered_final"]["original"], 10)
)

display(
    assertion_terminal(dataset["tabular"]["clustered_final"]["exploratory"], 2) and \
    assertion_terminal(dataset["tabular"]["clustered_final"]["exploratory"], 6)
)

display(
    assertion_terminal(dataset["tabular"]["clustered_final"]["evaluation"], 3) and \
    assertion_terminal(dataset["tabular"]["clustered_final"]["evaluation"], 43)
)

True

True

True

# Continuous

In [45]:
dataset["continuous"] = {}

In [ ]:
model["continuous"] = Model(5, 1024)
model["continuous"].load_weights( os.path.join(dir_policy["continuous"]["neural"], f"policy.h5"))

## Neural

### Datset

In [ ]:
def dataset_raw_to_RL_split(dataset_raw):
    RL_split = AmsterdamFormatter().RL_split(dataset_raw)
    return RL_split

In [ ]:
def dataset_raw_to_dataset_primal(dataset_raw):
    RL_split = dataset_raw_to_RL_split(dataset_raw)
    dataset_primal = RL_split_to_dataset_primal(RL_split, tabular=False)

    return dataset_primal

In [ ]:
head_by_id(
    dataset_raw_to_dataset_primal(
        load_datasets_raw()["test"]
    ),
    n=2,
)

,id,t,obs_init,obs,act,rew,obs_next
0,20,0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254..."
1,20,1,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254..."
2,20,2,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254..."
3,20,3,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254..."
4,20,4,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254..."
5,20,5,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254..."
6,20,6,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254..."
7,25,0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",2,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254..."
8,25,1,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",2,1.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254..."


In [50]:
# final processing

def get_probs_AmsterdamA2C(observation):
    o = observation
    o = list(o)
    o = tf.convert_to_tensor(o, tf.float32)

    l, _ = model["continuous"](o)
    p = tf.nn.softmax(l, axis=1)
    p = np.array(p)

    return p

def add_probs_continuous(df):
    df = df.copy(deep=False)
    df["probs_init"] = list( get_probs_AmsterdamA2C(df["obs_init"]) )
    df["probs"]      = list( get_probs_AmsterdamA2C(df["obs"]) )
    df["probs_next"] = list( get_probs_AmsterdamA2C(df["obs_next"]) )
    return df

dataset_primal_to_dataset_final = get_processing([
    add_probs_continuous,
    add_terminal_transition,
])

In [51]:
head_by_id(
    dataset_primal_to_dataset_final(
        dataset_raw_to_dataset_primal(
            load_datasets_raw()["test"]
        )
    ),
    n=2,
)

,id,t,obs_init,obs,act,rew,obs_next,probs_init,probs,probs_next
0,20,0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.6153204e-07, 0.41843063, 0.09803962, 0.2416..."
1,20,1,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.6153204e-07, 0.41843063, 0.09803962, 0.2416...","[1.3341968e-18, 0.99022615, 0.0011949288, 0.00..."
2,20,2,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[1.3341968e-18, 0.99022615, 0.0011949288, 0.00...","[4.3437872e-23, 0.99479574, 5.7580583e-06, 0.0..."
3,20,3,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.3437872e-23, 0.99479574, 5.7580583e-06, 0.0...","[9.325621e-10, 0.00401068, 0.008864737, 0.9811..."
4,20,4,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[9.325621e-10, 0.00401068, 0.008864737, 0.9811...","[4.656923e-15, 6.592461e-05, 0.0002901405, 0.9..."
5,20,5,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.656923e-15, 6.592461e-05, 0.0002901405, 0.9...","[2.4188708e-07, 8.889674e-06, 0.00064626464, 0..."
6,20,6,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[2.4188708e-07, 8.889674e-06, 0.00064626464, 0...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487..."
7,20,7,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487..."
8,25,0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",2,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[1.7647663e-11, 1.2354242e-06, 0.002096404, 0...."
9,25,1,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",2,1.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[1.7647663e-11, 1.2354242e-06, 0.002096404, 0....","[5.115847e-25, 0.00045865908, 0.0034875793, 0...."


In [58]:
def assertion_probs(df, column, tol=1e-6):
    for i in range( len(df) ):
        S = np.sum(df.loc[i, column])
        assert np.abs(S-1) < tol, S

In [59]:
def get_dataset_continuous():
    datasets_raw = load_datasets_raw()
    dataset_raw  = datasets_raw["test"]

    dataset_primal = dataset_raw_to_dataset_primal(dataset_raw)
    dataset_final  = dataset_primal_to_dataset_final(dataset_primal)

    for column in ["probs_init", "probs", "probs_next"]:
        assertion_probs(dataset_final, column)

    dataset_continuous = {
        "raw": dataset_raw,
        "primal": dataset_primal,
        "final": dataset_final,
    }

    return dataset_continuous

In [60]:
dataset["continuous"]["neural"] = get_dataset_continuous()

In [61]:
dataset["continuous"]["neural"]["final"].to_parquet(
    os.path.join(dir_dataset["continuous"], f"dataset.parquet"), )

In [62]:
head_by_id(dataset["continuous"]["neural"]["final"], n=2)

,id,t,obs_init,obs,act,rew,obs_next,probs_init,probs,probs_next
0,20,0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.6153204e-07, 0.41843063, 0.09803962, 0.2416..."
1,20,1,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.6153204e-07, 0.41843063, 0.09803962, 0.2416...","[1.3341968e-18, 0.99022615, 0.0011949288, 0.00..."
2,20,2,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[1.3341968e-18, 0.99022615, 0.0011949288, 0.00...","[4.3437872e-23, 0.99479574, 5.7580583e-06, 0.0..."
3,20,3,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.3437872e-23, 0.99479574, 5.7580583e-06, 0.0...","[9.325621e-10, 0.00401068, 0.008864737, 0.9811..."
4,20,4,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[9.325621e-10, 0.00401068, 0.008864737, 0.9811...","[4.656923e-15, 6.592461e-05, 0.0002901405, 0.9..."
5,20,5,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.656923e-15, 6.592461e-05, 0.0002901405, 0.9...","[2.4188708e-07, 8.889674e-06, 0.00064626464, 0..."
6,20,6,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[2.4188708e-07, 8.889674e-06, 0.00064626464, 0...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487..."
7,20,7,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487..."
8,25,0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",2,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[1.7647663e-11, 1.2354242e-06, 0.002096404, 0...."
9,25,1,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",2,1.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[1.7647663e-11, 1.2354242e-06, 0.002096404, 0....","[5.115847e-25, 0.00045865908, 0.0034875793, 0...."


In [63]:
assertion_terminal(dataset["continuous"]["neural"]["final"], 7) and \
assertion_terminal(dataset["continuous"]["neural"]["final"], 10)

True

### Bounds

In [ ]:
def load_bounds_raw():
    return AmsterdamFormatter() \
        .load_bounds(dir_split)

In [ ]:
load_bounds_raw()

,ID,Length of Stay,ACTH max,ACTH mean,ACTH min,ACTH std,AF Monitor max,AF Monitor mean,AF Monitor min,AF Monitor std,...,Fluid balance,AdmissionCount,Steroids,SOFA score,Age,Gender,Antibiotics,Antiviral,Antifungal,ICU Mortality
0,23545.0,1000,56.942379,27.410480,27.405037,2.410966,8.480442,14.704375,14.351298,2.343781,...,14.141715,11.421480,1.0,3.787396,5,2,3,1,1,1
1,11.0,0,-0.806916,-0.954016,-0.953790,-0.025408,-2.328308,-2.446705,-1.451279,-1.456343,...,-6.285162,-0.271663,0.0,-1.969128,0,0,0,0,0,0


In [ ]:
def bounds_raw_to_bounds_final(bounds_raw):
    bounds_raw_dropped = bounds_raw.drop(
        columns=["ID", "Length of Stay", "Steroids", "ICU Mortality"])

    bounds = {
        "obs_max": np.array(bounds_raw_dropped.loc[0]),
        "obs_min": np.array(bounds_raw_dropped.loc[1]),
    }

    return bounds

In [ ]:
bounds_raw_to_bounds_final(load_bounds_raw())

{'obs_min': array([ 5.69423790e+01,  2.74104805e+01,  2.74050369e+01,  2.41096640e+00,
         8.48044205e+00,  1.47043753e+01,  1.43512983e+01,  2.34378076e+00,
         2.12411346e+01,  2.77196846e+01,  2.83973942e+01,  1.25085926e+00,
         4.95312119e+00,  5.04160357e+00,  5.04312611e+00,  3.50155783e+00,
         3.08868542e+01,  3.29250221e+01,  3.44894981e+01, -2.00814128e+00,
         2.74480343e+01,  2.14069958e+01,  2.07772083e+01,  5.10954618e-01,
         8.80081081e+00,  1.09015923e+01,  1.79302856e-01,  1.64076187e+02,
         2.00921219e+02,  2.06600082e+02,  1.00391424e+00,  5.10706177e+01,
         4.57524920e+00,  5.58103991e+00,  4.78265643e-01,  2.65454044e+01,
         2.70125237e+01,  2.74020386e+01,  2.39406729e+00,  5.19698048e+00,
         5.49257851e+00,  5.34956837e+00, -5.68367660e-01,  9.14467087e+01,
         2.24091644e+02,  8.96394897e+02, -4.63353023e-02,  1.30696144e+01,
         1.30436869e+01,  1.30009518e+01, -4.32487205e-02,  7.78269434e+00,
 

In [ ]:
def assertion_bounds_final(bounds_final):
    return np.all(bounds_final["obs_min"] <= bounds_final["obs_max"])

In [ ]:
def get_bounds_continous():
    bounds_raw = load_bounds_raw()
    bounds_final = bounds_raw_to_bounds_final(bounds_raw)

    assert assertion_bounds_final(bounds_final)

    return {
        "raw": bounds_raw,
        "final": bounds_final,
    }

In [ ]:
bounds["continuous"] = get_bounds_continous()

In [ ]:
# np.save( os.path.join(dir_dataset["continuous"], f"obs_min.npy"), bounds["continuous"]["final"]["obs_min"], )
# np.save( os.path.join(dir_dataset["continuous"], f"obs_max.npy"), bounds["continuous"]["final"]["obs_max"], )

## Tabular

In [77]:
head_by_id(dataset["continuous"]["neural"]["final"], n=2)

,id,t,obs_init,obs,act,rew,obs_next,probs_init,probs,probs_next
0,20,0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.6153204e-07, 0.41843063, 0.09803962, 0.2416..."
1,20,1,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.6153204e-07, 0.41843063, 0.09803962, 0.2416...","[1.3341968e-18, 0.99022615, 0.0011949288, 0.00..."
2,20,2,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[1.3341968e-18, 0.99022615, 0.0011949288, 0.00...","[4.3437872e-23, 0.99479574, 5.7580583e-06, 0.0..."
3,20,3,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.3437872e-23, 0.99479574, 5.7580583e-06, 0.0...","[9.325621e-10, 0.00401068, 0.008864737, 0.9811..."
4,20,4,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[9.325621e-10, 0.00401068, 0.008864737, 0.9811...","[4.656923e-15, 6.592461e-05, 0.0002901405, 0.9..."
5,20,5,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.656923e-15, 6.592461e-05, 0.0002901405, 0.9...","[2.4188708e-07, 8.889674e-06, 0.00064626464, 0..."
6,20,6,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[2.4188708e-07, 8.889674e-06, 0.00064626464, 0...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487..."
7,20,7,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",1,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487..."
8,25,0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",2,0.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[1.7647663e-11, 1.2354242e-06, 0.002096404, 0...."
9,25,1,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[-0.0800016, -0.08865834, -0.08860586, -0.0254...",2,1.0,"[-0.0800016, -0.08865834, -0.08860586, -0.0254...","[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[1.7647663e-11, 1.2354242e-06, 0.002096404, 0....","[5.115847e-25, 0.00045865908, 0.0034875793, 0...."


In [76]:
head_by_id(dataset["tabular"]["clustered_final"]["original"], n=2)

,id,t,obs_init,obs,act,rew,obs_next,probs_init,probs,probs_next,probs_behavior_init,probs_behavior,probs_behavior_next
0,20,0,58,58,1,0.0,179,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.0009460958, 0.98486024, 0.00033117103, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.5987261, 0.18152866, 0.05732484, 0.10509554..."
1,20,1,58,179,1,0.0,17,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.0009460958, 0.98486024, 0.00033117103, 0.00...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.5987261, 0.18152866, 0.05732484, 0.10509554...","[0.64451313, 0.1576507, 0.038639877, 0.1112828..."
2,20,2,58,17,1,0.0,17,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64451313, 0.1576507, 0.038639877, 0.1112828...","[0.64451313, 0.1576507, 0.038639877, 0.1112828..."
3,20,3,58,17,1,0.0,17,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64451313, 0.1576507, 0.038639877, 0.1112828...","[0.64451313, 0.1576507, 0.038639877, 0.1112828..."
4,20,4,58,17,1,0.0,17,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64451313, 0.1576507, 0.038639877, 0.1112828...","[0.64451313, 0.1576507, 0.038639877, 0.1112828..."
5,20,5,58,17,1,0.0,17,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64451313, 0.1576507, 0.038639877, 0.1112828...","[0.64451313, 0.1576507, 0.038639877, 0.1112828..."
6,20,6,58,17,1,0.0,256,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.002630175, 0.0006064938, 0.0013344456, 0.00...","[0.099401385, 0.08730502, 0.018886177, 0.36740...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.64451313, 0.1576507, 0.038639877, 0.1112828...","[0.2, 0.2, 0.2, 0.2, 0.2]"
7,20,7,58,256,4,0.0,256,"[0.016237022, 0.0014849559, 0.6178519, 0.30005...","[0.099401385, 0.08730502, 0.018886177, 0.36740...","[0.099401385, 0.08730502, 0.018886177, 0.36740...","[0.64220184, 0.09633028, 0.07798165, 0.1422018...","[0.2, 0.2, 0.2, 0.2, 0.2]","[0.2, 0.2, 0.2, 0.2, 0.2]"
8,25,0,254,254,2,0.0,254,"[0.03472524, 0.033255864, 0.9320189, 0.0, 0.0]","[0.03472524, 0.033255864, 0.9320189, 0.0, 0.0]","[0.03472524, 0.033255864, 0.9320189, 0.0, 0.0]","[0.6666667, 0.16666667, 0.16666667, 0.0, 0.0]","[0.6666667, 0.16666667, 0.16666667, 0.0, 0.0]","[0.6666667, 0.16666667, 0.16666667, 0.0, 0.0]"
9,25,1,254,254,2,1.0,257,"[0.03472524, 0.033255864, 0.9320189, 0.0, 0.0]","[0.03472524, 0.033255864, 0.9320189, 0.0, 0.0]","[0.019111522, 0.31436116, 0.07228454, 0.422952...","[0.6666667, 0.16666667, 0.16666667, 0.0, 0.0]","[0.6666667, 0.16666667, 0.16666667, 0.0, 0.0]","[0.2, 0.2, 0.2, 0.2, 0.2]"


In [83]:
def get_dataset_hybrid(df_c, df_t):
    df = df_c.copy(deep=False)
    for label in ["obs_init", "obs", "obs_next"]:
        df[label] = df_t[label]

    return df

In [82]:
dataset["continuous"]["tabular"] = get_dataset_hybrid(
    dataset["continuous"]["neural"]["final"],
    dataset["tabular"]["clustered_final"]["original"],
)

In [84]:
head_by_id(dataset["continuous"]["tabular"], n=2)

,id,t,obs_init,obs,act,rew,obs_next,probs_init,probs,probs_next
0,20,0,58,58,1,0.0,179,"[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.6153204e-07, 0.41843063, 0.09803962, 0.2416..."
1,20,1,58,179,1,0.0,17,"[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.6153204e-07, 0.41843063, 0.09803962, 0.2416...","[1.3341968e-18, 0.99022615, 0.0011949288, 0.00..."
2,20,2,58,17,1,0.0,17,"[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[1.3341968e-18, 0.99022615, 0.0011949288, 0.00...","[4.3437872e-23, 0.99479574, 5.7580583e-06, 0.0..."
3,20,3,58,17,1,0.0,17,"[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.3437872e-23, 0.99479574, 5.7580583e-06, 0.0...","[9.325621e-10, 0.00401068, 0.008864737, 0.9811..."
4,20,4,58,17,1,0.0,17,"[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[9.325621e-10, 0.00401068, 0.008864737, 0.9811...","[4.656923e-15, 6.592461e-05, 0.0002901405, 0.9..."
5,20,5,58,17,1,0.0,17,"[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[4.656923e-15, 6.592461e-05, 0.0002901405, 0.9...","[2.4188708e-07, 8.889674e-06, 0.00064626464, 0..."
6,20,6,58,17,1,0.0,256,"[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[2.4188708e-07, 8.889674e-06, 0.00064626464, 0...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487..."
7,20,7,58,256,1,0.0,256,"[1.9520018e-30, 0.28778797, 0.007400115, 0.701...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487...","[0.0, 1.0, 9.294034e-10, 3.452601e-08, 7.56487..."
8,25,0,254,254,2,0.0,254,"[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[1.7647663e-11, 1.2354242e-06, 0.002096404, 0...."
9,25,1,254,254,2,1.0,257,"[4.83073e-08, 7.87445e-06, 0.07960352, 0.91251...","[1.7647663e-11, 1.2354242e-06, 0.002096404, 0....","[5.115847e-25, 0.00045865908, 0.0034875793, 0...."


In [88]:
dataset["continuous"]["tabular"].to_parquet(
    os.path.join(dir_policy["continuous"]["tabular"], "dataset.parquet")
)

# Common

In [440]:
onpe_continuous_primal = get_OnPV(dataset["continuous"]["primal"])
onpe_continuous_primal

0.013728592007028587

In [441]:
onpe_continuous_final = get_OnPV(dataset["continuous"]["final"])
onpe_continuous_final

0.013728592007028587

In [49]:
onpe_clustered_primal = get_OnPV(dataset["tabular"]["clustered_primal"]["test"])
onpe_clustered_primal

0.013728592007028587

In [50]:
onpe_clustered_final_original = get_OnPV(dataset["tabular"]["clustered_final"]["original"])
onpe_clustered_final_original

0.013728592007028587

In [51]:
onpe_clustered_final_exploratory = get_OnPV(dataset["tabular"]["clustered_final"]["exploratory"])
onpe_clustered_final_exploratory

0.017873921677523844

In [52]:
onpe_clustered_final_evaluation = get_OnPV(dataset["tabular"]["clustered_final"]["evaluation"])
onpe_clustered_final_evaluation

0.019905330552325445